# Grafana Logs - Data Preprocessing

This notebook performs preprocessing on Grafana logs from `ml_training_data.jsonl`.

## Key Issues with Grafana Logs:
1. **Mixed units**: Different panels use different units (percent, seconds, milliseconds, count)
2. **Different value ranges**: Metrics have vastly different scales
3. **Time-series nature**: Temporal patterns are important
4. **Categorical variables**: Need proper encoding
5. **Missing values**: Some fields like anomaly_type are null
6. **Class imbalance**: Likely more normal logs than anomalies

## Preprocessing Steps:
1. Load and explore the data
2. Handle missing values
3. Parse timestamps and extract temporal features
4. Normalize/scale numeric values per unit type
5. Handle categorical variables
6. Save preprocessed data

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 1. Load and Explore Data

In [ ]:
# Load JSONL data
data_path = '../../synthetic-log-generator/output/grafana/ml_training_data.jsonl'

logs = []
with open(data_path, 'r') as f:
    for line in f:
        logs.append(json.loads(line.strip()))

df = pd.DataFrame(logs)
print(f"Loaded {len(df)} log entries")
print(f"\nDataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Basic statistics
df.info()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0])
print(f"\nTotal missing values: {missing_values.sum()}")

In [ ]:
# Explore categorical columns
categorical_cols = ['dashboard', 'panel', 'service', 'unit', 'environment', 'cluster', 'datasource']

for col in categorical_cols:
    print(f"\n{col.upper()}:")
    print(f"Unique values: {df[col].nunique()}")
    print(df[col].value_counts().head(10))

In [ ]:
# Check anomaly distribution
print("Anomaly Distribution:")
print(df['is_anomaly'].value_counts())
print(f"\nAnomaly percentage: {df['is_anomaly'].mean() * 100:.2f}%")

if 'anomaly_type' in df.columns:
    print("\nAnomaly Types:")
    print(df[df['is_anomaly'] == True]['anomaly_type'].value_counts())

In [ ]:
# Visualize value distribution by unit
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Value Distribution by Unit Type', fontsize=16)

units = df['unit'].unique()[:4]
for idx, unit in enumerate(units):
    ax = axes[idx // 2, idx % 2]
    unit_data = df[df['unit'] == unit]['value']
    ax.hist(unit_data, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Unit: {unit}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Handle Missing Values

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Fill anomaly_type with 'normal' for non-anomalies
df_processed['anomaly_type'] = df_processed['anomaly_type'].fillna('normal')

# Check if metadata needs unpacking
if 'metadata' in df_processed.columns:
    # Extract metadata fields
    df_processed['panel_type'] = df_processed['metadata'].apply(lambda x: x.get('panel_type', 'unknown') if isinstance(x, dict) else 'unknown')
    df_processed['visualization'] = df_processed['metadata'].apply(lambda x: x.get('visualization', 'unknown') if isinstance(x, dict) else 'unknown')
    df_processed = df_processed.drop('metadata', axis=1)

print("Missing values handled!")
print(f"\nRemaining missing values: {df_processed.isnull().sum().sum()}")

## 3. Parse Timestamps and Extract Temporal Features

In [ ]:
# Convert timestamp to datetime
df_processed['timestamp'] = pd.to_datetime(df_processed['timestamp'])

# Extract temporal features
df_processed['hour'] = df_processed['timestamp'].dt.hour
df_processed['day_of_week'] = df_processed['timestamp'].dt.dayofweek
df_processed['day_of_month'] = df_processed['timestamp'].dt.day
df_processed['is_weekend'] = df_processed['day_of_week'].isin([5, 6]).astype(int)
df_processed['is_business_hours'] = df_processed['hour'].between(9, 17).astype(int)

# Time since start (in hours)
df_processed['hours_since_start'] = (df_processed['timestamp'] - df_processed['timestamp'].min()).dt.total_seconds() / 3600

print("Temporal features extracted:")
print(df_processed[['timestamp', 'hour', 'day_of_week', 'is_weekend', 'is_business_hours']].head())

In [ ]:
# Visualize temporal patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Temporal Patterns in Logs', fontsize=16)

# Hour distribution
df_processed['hour'].hist(bins=24, ax=axes[0, 0], edgecolor='black')
axes[0, 0].set_title('Distribution by Hour of Day')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Count')

# Day of week distribution
df_processed['day_of_week'].hist(bins=7, ax=axes[0, 1], edgecolor='black')
axes[0, 1].set_title('Distribution by Day of Week')
axes[0, 1].set_xlabel('Day (0=Monday)')
axes[0, 1].set_ylabel('Count')

# Anomaly by hour
anomaly_by_hour = df_processed.groupby('hour')['is_anomaly'].mean() * 100
anomaly_by_hour.plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Anomaly Percentage by Hour')
axes[1, 0].set_xlabel('Hour')
axes[1, 0].set_ylabel('Anomaly %')

# Time series of logs
df_processed.set_index('timestamp')['value'].resample('1H').count().plot(ax=axes[1, 1])
axes[1, 1].set_title('Log Count Over Time (Hourly)')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Normalize/Scale Numeric Values

Critical: Different units have different scales. We'll normalize within unit types.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Normalize values within each unit type using StandardScaler
df_processed['value_normalized'] = 0.0

for unit in df_processed['unit'].unique():
    mask = df_processed['unit'] == unit
    values = df_processed.loc[mask, 'value'].values.reshape(-1, 1)
    
    scaler = StandardScaler()
    normalized_values = scaler.fit_transform(values).flatten()
    
    df_processed.loc[mask, 'value_normalized'] = normalized_values

print("Value normalization completed!")
print(f"\nOriginal value stats:")
print(df_processed['value'].describe())
print(f"\nNormalized value stats:")
print(df_processed['value_normalized'].describe())

In [ ]:
# Compare original vs normalized values
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df_processed['value'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title('Original Value Distribution')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].set_yscale('log')

axes[1].hist(df_processed['value_normalized'], bins=100, edgecolor='black', alpha=0.7)
axes[1].set_title('Normalized Value Distribution')
axes[1].set_xlabel('Normalized Value')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 5. Create Statistical Features

For time-series data, rolling statistics can be very informative.

In [ ]:
# Sort by timestamp for rolling calculations
df_processed = df_processed.sort_values('timestamp').reset_index(drop=True)

# Calculate rolling statistics per service-panel combination
window_size = 10  # 10 consecutive measurements

df_processed['value_rolling_mean'] = df_processed.groupby(['service', 'panel'])['value_normalized'].transform(
    lambda x: x.rolling(window=window_size, min_periods=1).mean()
)

df_processed['value_rolling_std'] = df_processed.groupby(['service', 'panel'])['value_normalized'].transform(
    lambda x: x.rolling(window=window_size, min_periods=1).std().fillna(0)
)

df_processed['value_rolling_min'] = df_processed.groupby(['service', 'panel'])['value_normalized'].transform(
    lambda x: x.rolling(window=window_size, min_periods=1).min()
)

df_processed['value_rolling_max'] = df_processed.groupby(['service', 'panel'])['value_normalized'].transform(
    lambda x: x.rolling(window=window_size, min_periods=1).max()
)

# Deviation from rolling mean
df_processed['value_deviation'] = df_processed['value_normalized'] - df_processed['value_rolling_mean']

print("Statistical features created!")
print(df_processed[['value_normalized', 'value_rolling_mean', 'value_rolling_std', 'value_deviation']].head(15))

## 6. Save Preprocessed Data

In [ ]:
# Save preprocessed data
output_path = '../data/grafana_logs_preprocessed.csv'
df_processed.to_csv(output_path, index=False)
print(f"Preprocessed data saved to: {output_path}")
print(f"Shape: {df_processed.shape}")
print(f"\nColumns: {df_processed.columns.tolist()}")

In [ ]:
# Summary statistics
print("=" * 80)
print("PREPROCESSING SUMMARY")
print("=" * 80)
print(f"Total records: {len(df_processed):,}")
print(f"Date range: {df_processed['timestamp'].min()} to {df_processed['timestamp'].max()}")
print(f"Unique services: {df_processed['service'].nunique()}")
print(f"Unique dashboards: {df_processed['dashboard'].nunique()}")
print(f"Unique panels: {df_processed['panel'].nunique()}")
print(f"Anomaly rate: {df_processed['is_anomaly'].mean() * 100:.2f}%")
print(f"\nFeatures created: {len(df_processed.columns)}")
print("="  * 80)